In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, random_split
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# Descompresión minuciosa
with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

# Localización de ruta dinámica
data_path = '/content/dataset/seg_train/seg_train'
if not os.path.exists(data_path):
    data_path = '/content/dataset' # Ajuste por si la estructura varía

In [ ]:
# Transformaciones: Regularización y Buenas Prácticas
train_tfms = T.Compose([
    T.Resize((150, 150)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_tfms = T.Compose([
    T.Resize((150, 150)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = torchvision.datasets.ImageFolder(data_path)
classes = dataset.classes

# División didáctica del dataset
train_ds, test_ds = random_split(dataset, [int(0.8*len(dataset)), len(dataset)-int(0.8*len(dataset))])
train_ds.dataset.transform = train_tfms
test_ds.dataset.transform = test_tfms

dataloader = {
    'train': DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True),
    'test': DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)
}

print(f"Clases: {classes}")
print(f"Imágenes de entrenamiento: {len(train_ds)}")

In [ ]:
import random

ix = random.randint(0, len(train_ds))
img, label = train_ds[ix]

# Desnormalizar para mostrar imagen real
img_show = img.permute(1, 2, 0).numpy()
img_show = img_show * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]

plt.imshow(np.clip(img_show, 0, 1))
plt.title(f"Clase: {classes[label]}")
plt.axis('off')
plt.show()

In [ ]:
import torch.nn as nn

def block(c_in, c_out, k=3, p=1, s=1, pk=2, ps=2):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, k, padding=p, stride=s),
        nn.BatchNorm2d(c_out), # Optimización
        nn.ReLU(),
        nn.MaxPool2d(pk, stride=ps)
    )

class CNN(nn.Module):
    def __init__(self, n_channels=3, n_outputs=6):
        super().__init__()
        self.conv1 = block(n_channels, 64)
        self.conv2 = block(64, 128)
        self.conv3 = block(128, 256)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 18 * 18, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Regularización
            nn.Linear(256, n_outputs)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

model = CNN(n_outputs=len(classes))

In [ ]:
from tqdm import tqdm

def fit(model, dataloader, epochs=5):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    stats = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_acc = [], []
        bar = tqdm(dataloader['train'])
        for X, y in bar:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())
            acc = (y_hat.argmax(1) == y).sum().item() / len(y)
            train_acc.append(acc)
            bar.set_description(f"Epoch {epoch} Loss: {np.mean(train_loss):.4f} Acc: {np.mean(train_acc):.4f}")

        # Evaluación
        model.eval()
        val_loss, val_acc = [], []
        with torch.no_grad():
            for X, y in dataloader['test']:
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                loss = criterion(y_hat, y)
                val_loss.append(loss.item())
                acc = (y_hat.argmax(1) == y).sum().item() / len(y)
                val_acc.append(acc)

        print(f"Val Loss: {np.mean(val_loss):.4f} Val Acc: {np.mean(val_acc):.4f}")
        stats['train_loss'].append(np.mean(train_loss))
        stats['val_loss'].append(np.mean(val_loss))
        stats['val_acc'].append(np.mean(val_acc))
    return stats

history = fit(model, dataloader, epochs=10)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X, y in dataloader['test']:
        X = X.to(device)
        y_hat = model(X)
        y_true.extend(y.numpy())
        y_pred.extend(y_hat.argmax(1).cpu().numpy())

# Matriz de Confusión Didáctica
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Matriz de Confusión: Clasificación de Paisajes')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.show()

# Gráfica de Pérdida
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Historial de Pérdida')
plt.show()